# RAG

## Retrieving documents

### Manually

In [2]:
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [3]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [4]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [5]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [7]:
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.metadata)
print(post.content[:100])

{'title': 'Alerts', 'description': 'How to set up alerts.'}
<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [8]:
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [9]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [10]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")


Downloaded 95 documents


### With the `gitsource` library

In [11]:
!uv add gitsource

Resolved 114 packages in 1.85s                                       
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/8.64 KiB            
⠙ Preparing packages... (0/1)---------- 8.64 KiB/8.64 KiB           
Prepared 1 package in 137ms                                                       
Installed 1 package in 6ms                                  
 + gitsource==0.0.4


In [12]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [13]:
document = files[10]

print(document.filename)
print(document.content[:160])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:




## Parsing documents with the `frontmatter` library

In [14]:
import frontmatter 

post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename

In [15]:
document.parse()

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [16]:
documents = [f.parse() for f in files]

Each document now has:
- content - the actual documentation text
- metadata - title, description, and other fields
- filename - the original file path

## Indexing documents for search

We only want to add documents in the context that are relevant.

In [17]:
!uv add minsearch

Resolved 121 packages in 1.21s                                       
⠹ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠹ Preparing packages... (0/7)-------------------     0 B/301.83 KiB          
⠹ Preparing packages... (0/7)------------------- 16.00 KiB/301.83 KiB        
⠹ Preparing packages... (0/7)------------------- 32.00 KiB/301.83 KiB        
⠹ Preparing packages... (0/7)------------------- 40.13 KiB/301.83 KiB        
⠹ Preparing packages... (0/7)------------------- 40.13 KiB/301.83 KiB        
threadpoolctl        ------------------------------     0 B/18.20 KiB
⠹ Preparing packages... (0/7)------------------- 40.13 KiB/301.83 KiB        
threadpoolctl        ------------------------------     0 B/18.20 KiB
⠹ Preparing packages... (0/7)------------------- 40.13 KiB/301.83 KiB        
threadpoolctl        ------------------------------     0 B/18.20 KiB
⠹ Preparing p

In [18]:
from minsearch import Index

In [ ]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [ ]:
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [21]:
results

[{'title': 'LLM as a judge',
  'description': 'How to create and evaluate an LLM judge.',
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to cre